# Agent-X Pipeline Explorer
Interactive notebook for tracing questions through the vyakarana pipeline.

In [ ]:
import ast, json, sys
sys.path.insert(0, '.')
from upakarana.engine.client import Client

c = Client('/tmp/vy.sock')

def ev(expr):
    """Evaluate a tantra expression and return raw string."""
    return c.eval(expr)

def triples(raw):
    """Parse triple list from eval result."""
    if isinstance(raw, list): return raw
    return ast.literal_eval(raw)

def ask(q):
    """Ask a question, get the answer."""
    return ev(f'(anuvada-ganana "{q}")')

print('Connected:', ev('(node-count)'), 'nodes')

## Edge Type Dictionary
Every triple is `[subject, edge, object]`. Here's what each edge means in plain English.

In [ ]:
EDGES = {
    # ── BQG stage: word recognition ──
    'satya':              'known concept   — word resolved to a graph node',
    'mithya':             'unknown word    — not found in shabda dictionary',
    'asprista-sankhya':   'unbound number  — a number not yet assigned to a concept',
    'copula':             'is/are/was      — links subject to predicate ("mass IS 5")',
    'dvandva':            'and/comma       — conjunction boundary between clauses',
    'viraam':             'period/stop     — sentence boundary',
    'vidhi-kaala':        'intent verb     — find/calculate/determine (marks solve-for)',
    
    # ── Refine stage: binding ──
    'sankhya':            'bound number    — number assigned to its concept (mass=5)',
    'matra':              'unit            — physical unit (kg, m/s, joule)',
    'kaala':              'tense           — vartamana(present), bhuta(past), bhavishya(future)',
    'vachana':            'number/count    — eka(singular), bahu(plural)',
    'prathama-vibhakti':  'subject marker  — "X has Y" → X is prathama (nominative)',
    'shashthi-vibhakti':  'possession      — "X has Y" → Y belongs to X (genitive)',
    'sandhi-rename':      'compound join   — "kinetic" + "energy" → kinetic-energy',
    
    # ── Kosha-expand stage: knowledge graph ──
    'kosha-janya':        'PPR discovered  — related concept found via graph walk',
    
    # ── Signal stage ──
    '_signal':            'pipeline signal — dispatch-mode, solve-for, scope-entity',
    
    # ── Graph edges (visheshanam) ──
    'swarupa':            'is-a / type     — "electron is-a particle"',
    'abheda':             'same-as         — "speed same-as velocity"',
    'sthita':             'contains        — "varga contains momentum, energy"',
    'yukta':              'requires        — "force requires mass, acceleration"',
    'kriya':              'computes-via    — "momentum computes-via multiplication"',
    'phala':              'produces        — "multiplication produces product"',
    'janya':              'derived-from    — "kinetic-energy derived-from velocity"',
    'drishthanta':        'example         — "electron example-of particle"',
    'siddha':             'proven-by       — "conservation proven-by noether"',
    'pratipaksha':        'inverse-of      — "division inverse-of multiplication"',
    'krama':              'ordered-step    — mantra computation chain',
    'avastha':            'qualifier       — "kinetic" qualifies "energy"',
    'naama':              'word-for        — shabda word mapping',
    'lakshana':           'has-property    — "algebra has-property associativity"',
    'rahita':             'lacks           — "jada lacks karma (inert lacks action)"',
    'poorva':             'preceded-by     — temporal/logical ordering',
    'janaka':             'generates       — "vector-space generates subspace"',
}

def explain_edge(e):
    return EDGES.get(e, f'(unknown edge: {e})')

# Print as reference table
for e, desc in EDGES.items():
    print(f'  {e:25s} {desc}')

## Pipeline Trace
Change `Q` below and re-run to trace any question.

In [ ]:
Q = 'mass is 5 and velocity is 10. find kinetic energy'
# Q = 'force is 20 and mass is 4. find acceleration'
# Q = 'momentum is 30 and mass is 5. find velocity'
# Q = 'mass is 3 and velocity is 8. find momentum'

In [ ]:
# ── Stage 1: Build Question Graph ──
# Each word → shabda-anveshana (dictionary lookup) → triple
print(f'Q: {Q}')
print(f'{"─"*60}')
bqg = triples(ev(f'(build-question-graph "{Q}")'))
print(f'BQG: {len(bqg)} triples\n')
for s, e, o in bqg:
    print(f'  {s:20s}  ──{e:22s}──▶  {o:20s}  │ {explain_edge(e)}')

In [ ]:
# ── Stage 2: Refine ──
# Bind numbers to concepts, resolve compounds, extract grammar
ref = triples(ev(f'(avrti-refine-v2 (build-question-graph "{Q}"))'))
bqg_set = {str(t) for t in bqg}
new = [t for t in ref if str(t) not in bqg_set]
print(f'Refine: {len(bqg)} → {len(ref)} triples (+{len(new)} new)\n')
print('New bindings:')
for s, e, o in new:
    print(f'  + {s:20s}  ──{e:22s}──▶  {o:20s}  │ {explain_edge(e)}')

In [ ]:
# ── Stage 3: Kosha Expand ──
# PPR walk from satya nodes into knowledge graph
exp = triples(ev(f'(kosha-expand (avrti-refine-v2 (build-question-graph "{Q}")))'))
ref_set = {str(t) for t in ref}
kosha = [t for t in exp if str(t) not in ref_set]
print(f'Kosha: {len(ref)} → {len(exp)} triples (+{len(kosha)} kosha-janya)\n')

# Group by source concept
from collections import defaultdict
by_src = defaultdict(list)
for s, e, o in kosha:
    by_src[s].append(o)
for src, targets in sorted(by_src.items()):
    print(f'  {src:20s} → {targets[:8]}{" ..." if len(targets)>8 else ""}')

In [ ]:
# ── Stage 4: Detect Signals ──
sig = triples(ev(f'(detect-signals (kosha-expand (avrti-refine-v2 (build-question-graph "{Q}"))))'))
exp_set = {str(t) for t in exp}
signals = [t for t in sig if str(t) not in exp_set]
print('Signals detected:\n')
for s, e, o in signals:
    names = {'derive': 'formula computation', 'shunya': 'zero/absence reasoning',
             'viveka': 'comparison', 'count': 'arithmetic', 'anumana': 'categorical logic'}
    label = names.get(o, o)
    print(f'  {s:20s}  =  {o:25s}  │ {label}')

In [ ]:
# ── Stage 5: Dispatch + Mantra Match ──
pipe = f'(detect-signals (kosha-expand (avrti-refine-v2 (build-question-graph "{Q}"))))'
mode = ev(f'(read-signal {pipe} "dispatch-mode")')
sf   = ev(f'(read-signal {pipe} "solve-for")')
mm   = ev(f'(match-mantra {pipe})')
print(f'Dispatch mode:  {mode}')
print(f'Solve for:      {sf}')
print(f'Mantra match:   {mm}\n')

# ── Final answer ──
result = ask(Q)
print(f'\n{"═"*60}')
print(f'ANSWER: {result}')
print(f'{"═"*60}')

---
## Inspect Tools
Explore the graph directly — nodes, edges, shabda, mantras.

In [ ]:
def inspect(node):
    """Full node inspection: layer, edges, shabda."""
    layer = ev(f'(node-layer "{node}")')
    print(f'\n{node} (layer: {layer})')
    print(f'{"─"*50}')
    
    # Edges via walk
    for rel in ['swarupa','abheda','sthita','yukta','kriya','phala','janya',
                'pratipaksha','krama','avastha','naama','lakshana']:
        targets = ev(f'(walk "{node}" "{rel}")')
        if targets and targets != '[]' and targets != 'NIL':
            print(f'  {rel:20s} → {targets}')
    
    # Shabda keys
    for key in ['word','name','eval','arity','krama-lhs','krama-rhs','unit',
                'krama-lhs-unit','role','concepts-for-unit']:
        val = ev(f'(shabda "{node}" "{key}")')
        if val and val != 'NIL':
            print(f'  shabda.{key:15s} = {val}')

# Try it:
inspect('kinetic-energy')

In [ ]:
def inspect_mantra(name):
    """Show how a mantra computes: inputs → operation chain → output."""
    lhs = ev(f'(shabda "{name}" "krama-lhs")')
    rhs = ev(f'(shabda "{name}" "krama-rhs")')
    unit = ev(f'(shabda "{name}" "unit")')
    krama = ev(f'(node-krama "{name}")')
    
    print(f'\n{name}')
    print(f'{"─"*50}')
    print(f'  inputs:    {rhs}')
    print(f'  output:    {lhs}')
    print(f'  unit:      {unit}')
    print(f'  krama:     {krama}')
    
    # Walk the krama chain
    krama_edges = ev(f'(walk "{name}" "krama")')
    print(f'  chain:     {krama_edges}')

# Try it:
inspect_mantra('kinetic-energy-mantra')

In [ ]:
def word_lookup(word):
    """What does a word resolve to?"""
    node = ev(f'(shabda-anveshana "{word}")')
    if not node or node == 'NIL':
        print(f'  "{word}" → (unknown)')
        return
    layer = ev(f'(node-layer "{node}")')
    print(f'  "{word}" → {node} (layer: {layer})')

# Word lookup examples
words = ['mass', 'velocity', 'force', 'energy', 'momentum', 'acceleration',
         'what', 'find', 'calculate', 'is', 'has', 'not', 'and', 'the',
         'faster', 'heavier', 'at', 'rest', 'from']
print('Word → Node mapping:\n')
for w in words:
    word_lookup(w)

In [ ]:
def walk(node, rel, reverse=False):
    """Walk edges from a node."""
    if reverse:
        result = ev(f'(walk-in "{node}" "{rel}")')
    else:
        result = ev(f'(walk "{node}" "{rel}")')
    print(f'  {node} ──{rel}──▶ {result}')
    return result

# Example: what does the kinetic energy varga contain?
print('Forward walk:')
walk('mechanical-energy-varga', 'sthita')
print('\nReverse walk (what contains kinetic-energy?):')
walk('kinetic-energy', 'sthita', reverse=True)

---
## Quick Questions
Restart server first if session state is stale: `!.venv/bin/python3 -m upakarana vy stop; .venv/bin/python3 -m upakarana vy start`

In [ ]:
# ── Physics questions that work ──
questions = [
    'mass is 5 and velocity is 10. find kinetic energy',
    'mass is 3 and velocity is 8. find momentum',
    'force is 20 and mass is 4. find acceleration',
    'mass is 2 and acceleration is 9.8. find force',
    'mass is 10 and height is 5 and gravitational acceleration is 9.8. find potential energy',
]
for q in questions:
    r = ask(q)
    # Extract just the numeric answer
    if 'find:' in r:
        ans = r.split('find:')[-1].strip()
    else:
        ans = r[-50:]
    print(f'  Q: {q}')
    print(f'  A: {ans}\n')

---
## Composed Functions
Tantra expressions you can evaluate directly.

In [ ]:
# ── Direct eval: tantra s-expressions ──

# Math operations
print('Math:')
print('  add:', ev('(add 3 4)'))                    # 7
print('  mul:', ev('(mul 5 6)'))                    # 30
print('  div:', ev('(div 10 3)'))                   # 3.333
print('  sub:', ev('(sub 10 3)'))                   # 7

# List operations
print('\nLists:')
print('  map:    ', ev('(map [1 2 3] (fn x -> (mul x 2)))'))          # [2,4,6]
print('  filter: ', ev('(filter [1 2 3 4 5] (fn x -> (gt x 3)))'))   # [4,5]
print('  reduce: ', ev('(reduce [1 2 3 4] 0 (fn acc x -> (add acc x)))'))  # 10

# String ops
print('\nStrings:')
print('  split:  ', ev('(split "hello world" " ")'))
print('  concat: ', ev('(concat "hello" "-" "world")'))

# Graph queries
print('\nGraph:')
print('  node count: ', ev('(node-count)'))
print('  walk:       ', ev('(walk "force" "yukta")'))
print('  shabda:     ', ev('(shabda "mass" "word")'))

In [ ]:
# ── Execute a mantra directly ──
# Stack machine: push args, apply ops from krama chain

mantras = [
    ('kinetic-energy-mantra',  {'mass': '5', 'velocity': '10'}),  # 250
    ('momentum-mantra',        {'mass': '3', 'velocity': '8'}),   # 24
    ('force-mantra',           {'mass': '10', 'acceleration': '2'}),  # 20
]

for name, inputs in mantras:
    lhs = ev(f'(shabda "{name}" "krama-lhs")')
    args = ', '.join(f'{k}={v}' for k,v in inputs.items())
    # Build the input list for execute-mantra
    input_pairs = ' '.join(f'["{k}" "{v}"]' for k,v in inputs.items())
    result = ev(f'(execute-mantra "{name}" [{input_pairs}])')
    print(f'  {name}: {args} → {lhs} = {result}')

In [ ]:
# ── List all available mantras ──
import subprocess
r = subprocess.run(['.venv/bin/python3', '-m', 'upakarana', 'om', 'search', 'mantra'],
                   capture_output=True, text=True)
print('Available mantras:\n')
for line in r.stdout.strip().split('\n'):
    if 'mantra' in line.lower():
        print(f'  {line.strip()}')

In [ ]:
# ── Tantra source viewer ──
import subprocess

def tantra_source(name):
    """Print a tantra's source code."""
    r = subprocess.run(['.venv/bin/python3', '-m', 'upakarana', 'tantra', 'source', name],
                       capture_output=True, text=True)
    print(f'── {name}.tantra ──')
    print(r.stdout)

tantra_source('dispatch-derive')

---
## Helpful CLI Commands
Run these with `!` prefix in any cell:
```
# Server
!.venv/bin/python3 -m upakarana vy start
!.venv/bin/python3 -m upakarana vy stop
!.venv/bin/python3 -m upakarana vy reload

# Graph inspection
!.venv/bin/python3 -m upakarana om source <node>
!.venv/bin/python3 -m upakarana shabda node <node>
!.venv/bin/python3 -m upakarana vy inspect <node>
!.venv/bin/python3 -m upakarana vy walk <node> <relation>

# Analysis
!.venv/bin/python3 -m upakarana a hubs
!.venv/bin/python3 -m upakarana a dataflow
!.venv/bin/python3 -m upakarana a compose
!.venv/bin/python3 -m upakarana tantra callgraph

# Tests
!.venv/bin/python3 -m upakarana test run
!.venv/bin/python3 -m upakarana cache summary
```